# 06 · Baza RAG od zera: indeks, wersje, aktualizacja

Baza wektorowa nie jest źródłem prawdy. Źródłem są wersjonowane dokumenty; indeks zawiera fragmenty, ich embeddingi i wskaźniki do plików. W tej lekcji tworzymy lokalną Chroma, aktualizujemy jeden fragment, usuwamy nieaktualną wersję i sprawdzamy cytat po hashu. Każde uruchomienie używa osobnego katalogu pod `.tmp/rag`, nie dotyka danych Live Example.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import hashlib
import json
import sys
import tempfile
from pathlib import Path

import chromadb
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, ollama_settings, section_records

## 1. Jawny kontrakt indeksu

ID `document_id:version:section` jest niezmienne dla konkretnej wersji; nowa wersja dostaje nowe ID. Hash treści wykrywa zmianę, a nazwa modelu embeddingowego identyfikuje przestrzeń wektorową. Nigdy nie mieszaj wektorów różnych modeli w jednej kolekcji. `access_scope` to metadana dydaktyczna; produkcyjna autoryzacja musi być wymuszana przez aplikację, nie przez model.

In [ ]:
records = section_records()
host, embed_model = ollama_settings()
scratch = ROOT / ".tmp" / "rag"
scratch.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="index-", dir=scratch))
db = chromadb.PersistentClient(path=str(workspace / "chroma"))
collection = db.create_collection("acme-teaching-v1", configuration={"hnsw": {"space": "cosine"}},
                                  metadata={"embedding_model": embed_model})
len(records), workspace.name, embed_model

## 2. Ingest: tekst i metadata obok wektora

Wektor powstaje z tekstu sekcji. `upsert` jest powtarzalny dla tego samego ID, ale gdy treść lub model się zmieni, embedding musi zostać przeliczony. Poniżej budujemy manifest z hashami; pełny system porówna go z poprzednim przebiegiem i usunie wycofane ID.

In [ ]:
vectors = embed_texts([row["text"] for row in records])
collection.upsert(ids=[row["id"] for row in records], embeddings=vectors.tolist(),
                  documents=[row["text"] for row in records],
                  metadatas=[{key: row[key] for key in ("document_id", "version", "heading", "source_path", "access_scope", "content_hash")} for row in records])
manifest = {row["id"]: {"hash": row["content_hash"], "model": embed_model} for row in records}
(workspace / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
collection.count()

## 3. Update i delete bez starego wektora

Symulujemy nową wersję *jednej sekcji* w osobnym pliku `.tmp`, nie modyfikujemy prawdziwego dokumentu. Nowy ID wskazuje wersję 2; stare ID usuwamy. Najpierw przygotuj plik i nowy wektor, potem upsert, delete i aktualizację manifestu. Te operacje nie są tu jedną transakcją: przy dużych zbiorach zbuduj osobną kolekcję i przełącz odczyt dopiero po kontroli całości.

In [ ]:
original = records[0]
revised = original["text"] + "\n\nTeaching revision: verify the reporting cutoff before comparing totals."
new_id = original["id"].replace(":v1:", ":v2:")
new_hash = hashlib.sha256(revised.encode("utf-8")).hexdigest()
revision_path = workspace / f"{original['document_id']}.v2.teaching-section.md"
revision_path.write_text(revised, encoding="utf-8")
new_metadata = {key: original[key] for key in ("document_id", "heading", "access_scope")}
new_metadata.update({"version": 2, "source_path": revision_path.relative_to(ROOT).as_posix(),
                     "content_hash": new_hash})
collection.upsert(ids=[new_id], embeddings=embed_texts([revised]).tolist(), documents=[revised], metadatas=[new_metadata])
collection.delete(ids=[original["id"]])
manifest.pop(original["id"])
manifest[new_id] = {"hash": new_hash, "model": embed_model}
(workspace / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
assert collection.get(ids=[original["id"]])["ids"] == []
assert collection.get(ids=[new_id])["ids"] == [new_id]
assert set(collection.get()["ids"]) == set(manifest)
collection.count(), revision_path.relative_to(ROOT).as_posix()

## 4. Retrieval i weryfikacja źródła

Chroma zwraca kandydata. Po pobraniu rekordu sprawdzamy hash **i zgodność z tekstem źródłowym** oraz to, że ID widnieje w aktualnym manifeście. Osobno pobieramy nową wersję po ID i porównujemy zapisany wektor z ponownie policzonym embeddingiem. Sam poprawny update nie gwarantuje wysokiego rankingu dla każdego pytania; wynik ANN sprawdzamy oddzielnie. Wersja 2 jest symulacją lokalną, więc nie wolno cytować jej jako rzeczywistej rewizji Acme.

In [ ]:
query = embed_texts(["In the teaching revision, what should I verify before comparing totals?"])[0]
hits = collection.query(query_embeddings=[query.tolist()], n_results=3,
                        where={"access_scope": "finance"}, include=["documents", "metadatas", "distances"])
source_texts = {row["id"]: row["text"] for row in section_records()}
source_texts[new_id] = revision_path.read_text(encoding="utf-8")
verified = []
for identity, text, metadata, distance in zip(hits["ids"][0], hits["documents"][0], hits["metadatas"][0], hits["distances"][0]):
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == metadata["content_hash"]
    assert text == source_texts[identity]
    assert manifest[identity]["hash"] == metadata["content_hash"]
    assert (ROOT / metadata["source_path"]).is_file()
    verified.append({"id": identity, "source": metadata["source_path"], "distance": round(distance, 3)})
stored_revision = collection.get(ids=[new_id], include=["documents", "embeddings", "metadatas"])
assert stored_revision["documents"][0] == revision_path.read_text(encoding="utf-8")
revision_vector = embed_texts([revised])[0]
assert np.allclose(stored_revision["embeddings"][0], revision_vector, atol=1e-5)
probe = collection.query(query_embeddings=[revision_vector.tolist()], n_results=3,
                         where={"access_scope": "finance"}, include=["distances"])
{"question_top3": verified, "same_text_probe": probe["ids"][0]}

Jeśli `same_text_probe` nie umieszcza nowej wersji na pierwszym miejscu, porównaj wynik Chroma z dokładnym cosinusem zapisanych wektorów i sprawdź stan indeksu po ponownym otwarciu. `get(id)` potwierdza zapis rekordu; `query()` bada jeszcze jakość wyszukiwania.

### Utrzymanie w praktyce

Zmiana dokumentu → wykryj hash → przelicz tylko zmienione fragmenty → usuń stare ID → odnotuj wersję indeksu. Zmiana modelu embeddingowego → przebuduj **cały** indeks w nowej kolekcji. Dodaj testy: liczba rekordów, osierocone ID, zgodność hashy, filtry dostępu, recall na stałych pytaniach. Monitoruj puste wyniki i drift pytań. Nie przechowuj sekretów w metadanych ani w promptcie.

**Ćwiczenie:** powtórz `upsert` i sprawdź, czy `count()` rośnie. Potem usuń dodatkową sekcję i sprawdź, że nie wraca w wyszukiwaniu.